In [9]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI


In [10]:

documents = load_faq_data()
index = build_index(documents)


In [11]:
print(f"Loaded {len(documents)} documents")

Loaded 1350 documents


In [12]:

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)


Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still open.


# Agents


In [14]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can join, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you check:\n- whether the course is still accepting students\n- the deadline to enroll\n- any prerequisites or approval needed\n- how to register\n\nIf you share the course name or a link, I can help you figure it out.'

In [15]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [17]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered just discovered can I join enrollment late registration"}', call_id='call_d2DDhdaAtoIsnX9mSeo7LDOS', name='search', type='function_call', id='fc_0b8b6dc2342421f1006a40054b3e1c81a1bf1e4a1d8dcc12d4', namespace=None, status='completed')]

In [18]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [19]:
from pprint import pprint
pprint(result_json)

('[\n'
 '  {\n'
 '    "id": "74eb249bbf",\n'
 '    "course": "llm-zoomcamp",\n'
 '    "section": "General Course-Related Questions",\n'
 '    "question": "I just discovered the course. Can I still join?",\n'
 '    "answer": "Yes, but if you want to receive a certificate, you need to '
 'submit your project while we\\u2019re still accepting submissions."\n'
 '  },\n'
 '  {\n'
 '    "id": "977bf7786c",\n'
 '    "course": "llm-zoomcamp",\n'
 '    "section": "General Course-Related Questions",\n'
 '    "question": "Course: I have registered for the LLM Zoomcamp. When can I '
 'expect to receive the confirmation email?",\n'
 '    "answer": "You don\'t need it. You\'re accepted. You can also just start '
 'learning and submitting homework (while the form is open) without '
 'registering. It is not checked against any registered list. Registration is '
 'just to gauge interest before the start date."\n'
 '  },\n'
 '  {\n'
 '    "id": "69d122f12e",\n'
 '    "course": "llm-zoomcamp",\n'
 '    "

In [20]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 29)

In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


# The Agentic Loop

In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [26]:
question = "I just discovered the course. Can I join it?"


In [27]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [28]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"can I still join course discovered the course late enrollment join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. You can also follow the course in a self-paced way, but certificates are only available for the live cohort.

If you'd like, I can also help you figure out how to start from the current point in the course. Any other areas you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted. You can also follow the course in a self-paced way, but certificates are only available for the live cohort.\n\nIf you'd like, I can also help you figure out how to start from the current point in the course. Any other areas you want to explore?"

In [29]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry relevant to “queen’s gambit,” so I’m not able to answer that from the course materials.

If you meant something course-related, please rephrase it with the course/topic name, and I can look it up. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry relevant to “queen’s gambit,” so I’m not able to answer that from the course materials.\n\nIf you meant something course-related, please rephrase it with the course/topic name, and I can look it up. Are there other areas you want to explore?'